# 모델 개발 전 데이터 셋 정리 과정
1. 중복된 화합물 개수 찾아내기
2. 중복된 화합물 중 label 정보가 일치하지 않는 화합물 찾기
3. smiles 코드가 없는 물질 제거하기
4. 최종 데이터 확인 (label의 밸런스 확인)

In [23]:
import pandas as pd

df = pd.read_excel('skin_irritation.xlsx', sheet_name=2)

In [24]:
df.columns

Index(['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name',
       'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient',
       'Concentration', 'Concentration_Units', 'Mixture', 'Species',
       'Reported_Strain', 'Strain', 'Sex', 'Assay', 'Endpoint', 'Response',
       'Response_Unit', 'Reference', 'SMILES', 'Preferred_Name', 'Synonyms',
       'URL_CompTox', 'URL_CEBS'],
      dtype='str')

In [25]:
df_selec = df[(df['Mixture']=='Chemical') & (df['Species']=='Human')] #Mixture 컬럼이 Chemical, Species 컬럼은 Human인 경우 선택.

In [26]:
df_selec['Endpoint'].value_counts()

Endpoint
Qualitative classification    90
Positive reaction             90
Name: count, dtype: int64

In [27]:
df_class = df_selec[df_selec['Endpoint']=='Qualitative classification']

In [28]:
df_react = df_selec[df_selec['Endpoint']=='Positive reaction']

In [29]:
df_class['Response'].value_counts()

Response
Not classified                      65
Irritant                            14
Not classified/Possible irritant     8
Irritant/Possible corrosive          3
Name: count, dtype: int64

In [30]:
# Qualitatie classification 정리
df_class['label']=0  #label을 0으로 먼저 초기화하고,,,
df_class.loc[df_class['Response']!='Not classified','label']=1  #Not classified가 아니라면 (!=) label을 1로 설정.

In [31]:
# Positive reaction 정리
df_react['label']=0
df_react.loc[df_react['Response']!='0','label']=1 #Respons 값이 '0'이 아니라면 (!=) label을 1로 설정.

# Quiz1
왜 label을 정의할 때, Response 컬럼을 굳이 numerical value로 바꾸지 않아도 될까요?

In [22]:
pandas 입장에서는 Response 컬럼굳이 숫자형일 필요가 없기 때문입니다. 머신러닝 모델에 넣을 때만 숫자/원‑핫 인코딩이 필요한 것이고, “라벨” 그 자체는 문자열이어도 전혀 문제 없습니다.

SyntaxError: invalid character '‑' (U+2011) (1329398973.py, line 1)

In [32]:
df_total = pd.concat([df_class, df_react])
df_total['label'].value_counts()

label
1    115
0     65
Name: count, dtype: int64

# 데이터 중복 문제
1. 180개 데이터가 있지만, 화합물 개수는 중복되어 있을 수 있으니 확인 필요.
2. 중복된 화합물 중에서 label이 불일치 하는 경우가 있는지 확인 필요.

In [33]:
df_total['Chemical_Name']

0                  Alcohol ethoxylate C12-15/E5 phosphate
1           C12-13 beta-branched primary alco hol sulfate
2       C12-13 beta-branched primary alco hol sulfate/...
5                   N,N-Dimethyl-N-dodecyl amino- betaine
6       Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)],...
                              ...                        
1850                                      Triethanolamine
1852                                        Decanoic acid
1858                                       1-Tetradecanol
1860       Ethylenediaminetetraacetic acid, disodium salt
1862               Hexadecanoic acid, 1-methylethyl ester
Name: Chemical_Name, Length: 180, dtype: str

In [34]:
df_total['Chemical_Name'].to_list()

['Alcohol ethoxylate C12-15/E5 phosphate',
 'C12-13 beta-branched primary alco hol sulfate',
 'C12-13 beta-branched primary alco hol sulfate/1-ethoxylate',
 'N,N-Dimethyl-N-dodecyl amino- betaine',
 'Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex',
 'Alkyl polyglucoside 600',
 'Alcohol ethoxylate C16-18/E14',
 'Alcohol ethoxylate C16-18/E5',
 'Alcohol ethoxylate C12-15/E3',
 'Alcohol ethoxylate C11/E7',
 'Alcohol ethoxylate C11/E3',
 'Tween 80',
 'Tween 80',
 'Propylene glycol',
 'Heptanal',
 'Isopropyl myristate',
 'di-Propylene glycol',
 'Sodium hydroxide',
 'Sodium hydroxide',
 '4-Methylthio benzaldehyde',
 'Heptyl butyrate',
 'Hydroxycitronellal',
 'Methyl caproate',
 'Alkyl dimethyl betaine',
 '1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)',
 'Benzyl salicylate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium dodecyl sulphate',
 'Sodium carbonate',
 'Benzalkonium chloride',
 '1-Bro

In [35]:
chemical_name = df_total['Chemical_Name'].to_list()
unique_name = set(chemical_name) #set()을 사용하면, 중복된 item은 모두 삭제됩니다.

In [36]:
unique_name

{'(2E)-3,7-Dimethyl-2,6-octadien-1-ol',
 '1,2-Propylene glycol',
 '1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)',
 '1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)',
 '1-Bromo-4-chlorobutane',
 '1-Bromohexane',
 '1-Decanol',
 '1-Dodecanol',
 '1-Hexanol',
 '1-Naphthalene acetic acid',
 '1-Naphthaleneacetic acid',
 '1-Octanol',
 '1-Pentanol',
 '1-Spiro[4.5]dec-7-en-7-yl-4-penten-1-one',
 '1-Tetradecanol',
 '1-tert-Butoxy-2-propanol',
 '10-Undecenoic acid',
 '2-Isopropyl-2-isobutyl-1,3-dime- thoxypropane',
 '3,4-Dimethyl-1H-pyrazole',
 '4-(Methylthio)benzaldehyde',
 '4-Methylthio benzaldehyde',
 'Acetic acid',
 'Alcohol ethoxylate C11/E3',
 'Alcohol ethoxylate C11/E7',
 'Alcohol ethoxylate C12-15/E3',
 'Alcohol ethoxylate C12-15/E5 phosphate',
 'Alcohol ethoxylate C16-18/E14',
 'Alcohol ethoxylate C16-18/E5',
 'Alkyl dimethyl betaine',
 'Alkyl polyglucoside 600',
 'Amines, hydrogenated tallow alkyl',
 'Benzalkonium chloride',
 'Benzene, 1-(1-methylethyl)-2-

In [37]:
len(unique_name) #180개 데이터가 있었지만, 단일 물질 기준으로는 119개가 있네요.

119

In [38]:
for each in unique_name:
    print (each)

Butyl benzoate
Benzyl-C8-18-alkyldimethylammonium chlorides
Hexadecanoic acid
1-Tetradecanol
Dodecanol
Ethylenediamine tetraacetic acid disodium salt
Sodium dodecyl sulphate
Alcohol ethoxylate C16-18/E5
Benzyl alcohol
Terpinyl acetate
Hydrochloric acid
1-Naphthaleneacetic acid
1-Naphthalene acetic acid
di-n-Propyl disulfide
Methyl caproate
3,4-Dimethyl-1H-pyrazole
10-Undecenoic acid
Fatty acids, potassium salts
Citronellol
1-Bromo-4-chlorobutane
Octanoic acid
Sodium carbonate
Propylene glycol tertiary butyl ether
alpha-Terpinyl acetate
Methyl hexadecanoate
Potassium soap
Decanoic acid
Polyethylene glycol
Alcohol ethoxylate C11/E7
Methyl palmitate
Benzalkonium chloride
Bis[(1-Methylimidazol)-(2- ethyl- hexanoate)], zinc complex
Sodium percarbonate
1-tert-Butoxy-2-propanol
Methyl 2-((7-hydroxy-3,7-dimethyloctylidene)amino)benzoate
Carbonic acid sodium salt (1:2)
Dipropyl disulfide
Sodium xylene sulfonate
Dodecanoic acid
Tromethamine
4-(Methylthio)benzaldehyde
alpha-Terpineol
Butanols
Coc

In [39]:
#label 정보는 동일 화합물이라면 1이거나 0이어야 함. nunique()는 동일 화합물의 label에 있는 정보의 개수 확인.
#groupby(Chemical Name)은 화합물 이름을 기준으로 dataframe을 묶어봅니다.
#이름 기준으로 묶었을 때, label 정보가 몇개 있는지 확인하는 방식.
df_total.groupby('Chemical_Name')['label'].nunique()

Chemical_Name
(2E)-3,7-Dimethyl-2,6-octadien-1-ol                                 1
1,2-Propylene glycol                                                1
1-(2-Isopropylphenyl)-1-phenyle- thane (Mixture of isomers)         1
1-(Spiro[4.5]dec-7-en-7-yl)pent-4- en-1-one (mixture of isomers)    1
1-Bromo-4-chlorobutane                                              2
                                                                   ..
alpha-Terpineol                                                     2
alpha-Terpinyl acetate                                              1
di-Propylene glycol                                                 1
di-n-Propyl disulfide                                               1
n-Pentanol                                                          1
Name: label, Length: 119, dtype: int64

# Quiz2
df_total.groupby('Chemical_Name')['label'].nunique()를 실행했을 때, 도대체 왜 최대값은 2, 최소값은 1이 나올까요?

In [42]:
#데이터에 일부 화합물에서 label 값이 불일치하기 때문입니다.
#groupby('Chemical_Name')['label'].nunique()은 각 화합물별로 label 컬럼의 고유값 개수를 계산합니다.

#대부분 화합물: 같은 Chemical_Name에 label이 모두 1개 (nunique=1)

#일부 화합물: 같은 Chemical_Name에 label이 0과 1 둘 다 존재 (nunique=2)

SyntaxError: invalid syntax (1175267305.py, line 2)

In [43]:
df_total.groupby('Chemical_Name')['label'].nunique().max()

np.int64(2)

In [44]:
df_total.groupby('Chemical_Name')['label'].nunique().min()

np.int64(1)

In [45]:
df_total.groupby('DTXSID')['label'].nunique()

DTXSID
DTXSID0021175    2
DTXSID0021206    2
DTXSID0021597    1
DTXSID0026838    2
DTXSID0027856    2
                ..
DTXSID9021392    2
DTXSID9021554    1
DTXSID9026926    2
DTXSID9027073    2
DTXSID9027104    2
Name: label, Length: 70, dtype: int64

In [46]:
df_total.groupby('DTXSID')

In [47]:
import pandas as pd

# 당신의 원본 데이터 로드 (파일 경로 수정 필요)
df_total = pd.read_excel('skin_irritation.xlsx', sheet_name=2)
# 또는 df_total = pd.read_excel('your_file.xlsx')

print("df_total shape:", df_total.shape)
print("컬럼 확인:", df_total.columns.tolist())
print("'Chemical_Name', 'label' 컬럼 존재:", 'Chemical_Name' in df_total.columns, 'label' in df_total.columns)

# 원래 문제 확인
print("label 고유값 개수 통계:")
print(df_total.groupby('Chemical_Name')['label'].nunique().describe())

# 문제 화합물 찾기
problematic = df_total.groupby('Chemical_Name')['label'].nunique()
problem_compounds = problematic[problematic > 1].index.tolist()
print(f"\nlabel이 2개 이상인 화합물 {len(problem_compounds)}개")

# 첫 3개 화합물 상세 확인
if problem_compounds:
    print("\n문제 화합물 데이터:")
    print(df_total[df_total['Chemical_Name'].isin(problem_compounds[:3])])

df_total shape: (2058, 25)
컬럼 확인: ['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name', 'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient', 'Concentration', 'Concentration_Units', 'Mixture', 'Species', 'Reported_Strain', 'Strain', 'Sex', 'Assay', 'Endpoint', 'Response', 'Response_Unit', 'Reference', 'SMILES', 'Preferred_Name', 'Synonyms', 'URL_CompTox', 'URL_CEBS']
'Chemical_Name', 'label' 컬럼 존재: True False
label 고유값 개수 통계:


KeyError: 'Column not found: label'

In [48]:
# 중복 label 화합물만 필터링해서 확인
df_total[df_total.duplicated(subset=['Chemical_Name'], keep=False)].sort_values('Chemical_Name')

,Record_ID,Data_Type,Formulation_ID,Formulation_Name,Chemical_Name,CASRN,DTXSID,Percent_Active_Ingredient,Concentration,Concentration_Units,...,Assay,Endpoint,Response,Response_Unit,Reference,SMILES,Preferred_Name,Synonyms,URL_CompTox,URL_CEBS
815,skin_irritation_invivo_265,In Vivo,MIX374,Pyrocide Falcon 7452,"1,2,3,6-Tetrahydro-N-octyl-3,6-methanophthalimide",7786-80-3,DTXSID30999059,3.00,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,3,Unitless,FIFRA (undated),CCCCCCCCN1C(=O)C2C3CC(C=C3)C2C1=O,"1,2,3,6-Tetrahydro-N-octyl-3,6-methanophthalimide","7786-80-3|1,2,3,6-Tetrahydro-N-octyl-3,6-metha...",https://comptox.epa.gov/dashboard/chemical/det...,NaN
814,skin_irritation_invivo_266,In Vivo,MIX196,Evercide Total Release Fogger,"1,2,3,6-Tetrahydro-N-octyl-3,6-methanophthalimide",7786-80-3,DTXSID30999059,0.80,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,3,Unitless,FIFRA (undated),CCCCCCCCN1C(=O)C2C3CC(C=C3)C2C1=O,"1,2,3,6-Tetrahydro-N-octyl-3,6-methanophthalimide","7786-80-3|1,2,3,6-Tetrahydro-N-octyl-3,6-metha...",https://comptox.epa.gov/dashboard/chemical/det...,NaN
1128,skin_irritation_invivo_852,In Vivo,MIX6,Acticide CBM,"1,2-Benzisothiazolin-3-one",2634-33-5,DTXSID5032523,3.75,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,1,Unitless,FIFRA (undated),O=C1NSC2=C1C=CC=C2,"1,2-Benzisothiazolin-3-one","2634-33-5|1,2-Benzisothiazolin-3-one|1,2-benci...",https://comptox.epa.gov/dashboard/chemical/det...,NaN
1124,skin_irritation_invivo_519,In Vivo,MIX6,Acticide CBM,"1,2-Benzisothiazolin-3-one",2634-33-5,DTXSID5032523,3.75,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,1,Unitless,FIFRA (undated),O=C1NSC2=C1C=CC=C2,"1,2-Benzisothiazolin-3-one","2634-33-5|1,2-Benzisothiazolin-3-one|1,2-benci...",https://comptox.epa.gov/dashboard/chemical/det...,NaN
1125,skin_irritation_invivo_523,In Vivo,MIX11,Acticide SR 8213 C,"1,2-Benzisothiazolin-3-one",2634-33-5,DTXSID5032523,10.00,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,3,Unitless,FIFRA (undated),O=C1NSC2=C1C=CC=C2,"1,2-Benzisothiazolin-3-one","2634-33-5|1,2-Benzisothiazolin-3-one|1,2-benci...",https://comptox.epa.gov/dashboard/chemical/det...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1421,skin_irritation_invivo_434,In Vivo,MIX126,CSI Lambda 25 CS,lambda-Cyhalothrin,91465-08-6,DTXSID7032559,23.60,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,4,Unitless,FIFRA (undated),CC1(C)[C@H](C=C(/Cl)C(F)(F)F)[C@@H]1C(=O)O[C@@...,lambda-Cyhalothrin,91465-08-6|lambda-Cyhalothrin|415-130-7|A 1:1 ...,https://comptox.epa.gov/dashboard/chemical/det...,NaN
1423,skin_irritation_invivo_675,In Vivo,MIX95,Chemsico Aerosol LEG,lambda-Cyhalothrin,91465-08-6,DTXSID7032559,0.01,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,3,Unitless,FIFRA (undated),CC1(C)[C@H](C=C(/Cl)C(F)(F)F)[C@@H]1C(=O)O[C@@...,lambda-Cyhalothrin,91465-08-6|lambda-Cyhalothrin|415-130-7|A 1:1 ...,https://comptox.epa.gov/dashboard/chemical/det...,NaN
1416,skin_irritation_invivo_255,In Vivo,MIX186,Endigo ZCX,lambda-Cyhalothrin,91465-08-6,DTXSID7032559,9.59,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,3,Unitless,FIFRA (undated),CC1(C)[C@H](C=C(/Cl)C(F)(F)F)[C@@H]1C(=O)O[C@@...,lambda-Cyhalothrin,91465-08-6|lambda-Cyhalothrin|415-130-7|A 1:1 ...,https://comptox.epa.gov/dashboard/chemical/det...,NaN
24,skin_irritation_invivo_497,In Vivo,MIX86,Captevate 68 WDG,NaN,NaN,NaN,1.20,NaN,NaN,...,Draize Skin Irritation/Corrosion Test,EPA classification,4,Unitless,FIFRA (undated),NaN,NaN,NaN,NaN,NaN


In [53]:
label_counts = df_total.groupby('Chemical_Name')['label'].nunique()
bad_names = label_counts[label_counts > 1].index
print (bad_names)
print ('label 정보가 불일치하는 화합물 개수: ',len(bad_names))

KeyError: 'Column not found: label'

In [ ]:
df_filtered = df_total.groupby('Chemical_Name').filter(lambda x: x['label'].nunique() == 1)

print(f"전체: {df_total.shape}행")
print(f"label이 불일치 하는 화합물 제거: {df_filtered.shape}행")

In [ ]:
df_cleaned = df_filtered.drop_duplicates(subset='Chemical_Name')
print(f"label이 일치 하는 화합물 중 중복된 결과 제거: {df_cleaned.shape}행")

# Quiz3
1. smiles 코드가 없는 물질 이름을 확인해봅시다.
2. pubchem 데이터베이스 (https://pubchem.ncbi.nlm.nih.gov/)에서 물질 명을 입력해서 검색해봅시다.
3. 구글에서 물질 명을 입력해서 검색해봅시다.
4. CASRN (카스넘버)는 물질의 이름을 표시하는 또 다른 방법입니다. 동일 화학물질이 여러개의 이름을 가질 수 있어서, 고유한 번호를 부여해서 화합물을 찾아요.
5. df_cleaned에서 smiles 코드가 없는 물질들은 왜 없는지 검색을 통해 찾아봅시다.

In [ ]:
#smiles 코드가 없는 물질?
df_cleaned['SMILES']

In [ ]:
df_cleaned['SMILES'].isna()

In [ ]:
df_smi = df_cleaned.dropna(subset=['SMILES'])

In [ ]:
df_smi.shape

In [ ]:
df_smi['label'].value_counts()

In [ ]:
찾았는데 있어요.
Polysorbate 
SMILES : CCCCCCCCCCCC(=O)OCCOC[C@H]([C@@H]1[C@@H]([C@@H](CO1)OCCO)OCCO)OCCO
coco-betain
SMILES : CCCCCCCCCCCC(=O)NCCC[N+](C)(C)CC(=O)[O-]

In [ ]:
단일물질이 아니라 화합물이어서

In [ ]:
df_smi = df.copy() 

In [ ]:
# 1. 중복된 컬럼 제거 (이름 기준)
df_smi = df_smi.loc[:, ~df_smi.columns.duplicated()]

# 2. 만약 특정 개수(81개)만 정확히 남겨야 한다면 슬라이싱 확인
# (예: 앞에서부터 81개만 필요한 경우)
# df_smi = df_smi.iloc[:, :81]

# 3. 그 결과를 파일로 저장
df_smi.to_csv('w4-1_rdkit_cleaned.csv', index=False)

print(f"최종 컬럼 개수: {len(df_smi.columns)}")

In [ ]:
df_smi.to_csv('w4-1_rdkit.csv')